In [2]:
!hostname



c-11-03


In [3]:
import sys
import gzip
import glob
import numpy as np
import pandas as pd

import matplotlib
from matplotlib import pyplot as plt


In [4]:
def load_mgatk_output(output_dir, mito_length):
    # assuming mgatk output naming convention
    base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

    base_coverage_dict = dict()
    for i, nt in enumerate("ATCG"):
        cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

        # gather coverage for forward strand
        fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index=1, columns=0)
        fwd_base_df.columns = [x[1] for x in fwd_base_df.columns.values]  # flatten weird multiindex after pivot
        fwd_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in fwd_base_df.columns]
        # fwd_base_df[missing_pos] = 0  # fill in missing positions
        all_columns = [x for x in range(1, mito_length + 1)]
        fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
        fwd_base_df = fwd_base_df.fillna(0).sort_index(axis=1)  # assume all nan are true zeroes

        # gather coverage for forward strand
        rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
        rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
        rev_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
        # rev_base_df[missing_pos] = 0
        all_columns = [x for x in range(1, mito_length + 1)]
        rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
        rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

        # organize base data into a dict
        base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

    return base_coverage_dict


def gather_possible_variants(base_coverage_dict, reference_file):
    # sum across cells and strands for each base and position
    aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
    for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

    # make a reference set of tuples (pos, ref_base)
    ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
    ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
    ref_set = set([(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters])
    ref_dict = dict(ref_set)

    # make an observed set of tuples which are nonzero
    non_zero_idx = np.where(aggregated_genotype > 0)
    non_zero_bases = [letters[i] for i in non_zero_idx[0]]
    non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
    observed_set = list(zip(non_zero_pos, non_zero_bases))
    observed_set = set([x for x in observed_set if x[0] not in ref_N_positions])  # disregard positions in ref with N

    # take difference between observed and reference
    variant_set = observed_set - ref_set
    variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0])  # (pos, ref base, obs base)

    return variants


In [20]:
# sys.argv[1]
# MGATK_OUT_DIR = "/scr1/users/liuc9/mitochondrial/realdata/05-Liming/scmocha-celline/cromwell-executions/scMOCHA/4131ecab-f8d6-4654-89e8-5c82f148c1cf/call-call_mt_variants/execution/cell/final/"
MGATK_OUT_DIR = "/home/liuc9/github/scMOCHA/05-Liming/scmocha-mixed-cellline-high-depth2/cromwell-executions/scMOCHA/139358d8-df39-4274-b931-9c42b8d9c3bb/call-call_mt_variants/execution/cluster/final"
sample_prefix = "cluster"  # sys.argv[2]
mito_length = 16569  # int(sys.argv[3])  # 16569
low_coverage_threshold = 10  # int(sys.argv[4])  # 10
mito_genome = "MT"  # sys.argv[5]  # chrM

In [21]:
letters = list("ATCG")


In [22]:
base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length)
cell_barcodes = base_coverage_dict["A"][0].index

In [23]:
cell_barcodes

Index(['AAACCCAAGGCAGGTT-1', 'AAACCCACAACCAGAG-1', 'AAACCCACAAGCACAG-1',
       'AAACCCAGTGTTGATC-1', 'AAACCCATCACCATCC-1', 'AAACCCATCCAAATGC-1',
       'AAACCCATCCGATTAG-1', 'AAACCCATCCTAGCCT-1', 'AAACCCATCGCTGTTC-1',
       'AAACCCATCGTAATGC-1',
       ...
       'TTTGGTTTCAGACTGT-1', 'TTTGGTTTCAGCTCTC-1', 'TTTGTTGAGCAGATAT-1',
       'TTTGTTGCATGCAGCC-1', 'TTTGTTGCATTCCTAT-1', 'TTTGTTGGTGAACCGA-1',
       'TTTGTTGGTTTCCATT-1', 'TTTGTTGTCAGAATAG-1', 'TTTGTTGTCCTACCAC-1',
       'TTTGTTGTCTGGGCAC-1'],
      dtype='object', length=7862)

In [12]:
# total coverage per position per cell
total_coverage = pd.DataFrame(np.zeros((len(cell_barcodes), mito_length)), index=cell_barcodes, columns=np.arange(1, mito_length + 1))

for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]

In [17]:

# exclude low coverage cells from variant calling
cell_barcodes = total_coverage.index[total_coverage.mean(axis=1) > low_coverage_threshold]
for nt in base_coverage_dict:
    base_coverage_dict[nt] = (base_coverage_dict[nt][0].loc[cell_barcodes, :], base_coverage_dict[nt][1].loc[cell_barcodes, :])
total_coverage = total_coverage.loc[cell_barcodes, :]


In [19]:
cell_barcodes

Index([], dtype='object')

In [35]:
MGATK_OUT_DIR + mito_genome + "_refAllele.txt"

'/scr1/users/liuc9/mitochondrial/realdata/05-Liming/scmocha-celline/cromwell-executions/scMOCHA/4131ecab-f8d6-4654-89e8-5c82f148c1cf/call-call_mt_variants/execution/cell/final/MT_refAllele.txt'

In [18]:
base_coverage_dict["A"]

(Empty DataFrame
 Columns: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, ...]
 Index: []
 
 [0 rows x 16569 columns],
 Empty DataFrame
 Columns: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, ...]
 Index: []
 
 [0 rows x 16569 columns])

In [38]:
base_coverage_dict = base_coverage_dict
reference_file = MGATK_OUT_DIR + mito_genome + "_refAllele.txt"
aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum
ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
ref_set = set([(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters])
ref_dict = dict(ref_set)

# make an observed set of tuples which are nonzero
non_zero_idx = np.where(aggregated_genotype > 0)
non_zero_bases = [letters[i] for i in non_zero_idx[0]]
non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
observed_set = list(zip(non_zero_pos, non_zero_bases))
observed_set = set([x for x in observed_set if x[0] not in ref_N_positions])  # disregard positions in ref with N

# take difference between observed and reference
variant_set = observed_set - ref_set
variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0])


In [40]:
variants

[]

In [31]:

# call potential variants
variants = gather_possible_variants(base_coverage_dict, MGATK_OUT_DIR + mito_genome + "_refAllele.txt")
variant_names = ["{}{}>{}".format(x[0], x[1], x[2]) for x in variants]


In [32]:
variants

[]

In [ ]:

# build two <cell by variant tables>, one for each strand
total_coverage_variant_df = []
fwd_cell_variant_df, rev_cell_variant_df = [], []
for i, var in enumerate(variants):
    var_name = variant_names[i]
    pos, base = var[0], var[2]
    total_coverage_variant_df.append(total_coverage[pos])
    fwd_cell_variant_df.append(base_coverage_dict[base][0][pos].values)
    rev_cell_variant_df.append(base_coverage_dict[base][1][pos].values)
total_coverage_variant_df = pd.DataFrame(np.array(total_coverage_variant_df).T, index=cell_barcodes, columns=variant_names)
fwd_cell_variant_df = pd.DataFrame(np.array(fwd_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
rev_cell_variant_df = pd.DataFrame(np.array(rev_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
all_cell_variant_df = fwd_cell_variant_df + rev_cell_variant_df


In [ ]:
# load_mgatk_output

output_dir = MGATK_OUT_DIR
mito_lenght = mito_length

base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

base_coverage_dict = dict()

# for i, nt in enumerate("ATCG"):
#     cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)
i = 0

cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

In [ ]:
cur_base_data

In [ ]:
fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index = 1, columns = 0)
fwd_base_df.columns = [x[1] for x in fwd_base_df.columns.values]
fwd_base_df.index.name = None

In [ ]:
all_columns = [x for x in range(1, mito_length + 1)]
fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
fwd_base_df = fwd_base_df.fillna(0).sort_index(axis=1)

In [ ]:
# gather coverage for forward strand
rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
rev_base_df.index.name = None
# missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
# rev_base_df[missing_pos] = 0
all_columns = [x for x in range(1, mito_length + 1)]
rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

In [ ]:
base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

In [ ]:
base_coverage_dict

In [ ]:
base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length)


In [ ]:
cell_barcodes = base_coverage_dict["A"][0].index


In [ ]:
cell_barcodes


In [ ]:
# total coverage per position per cell

total_coverage = pd.DataFrame(
    np.zeros((len(cell_barcodes), mito_length)),
    index=cell_barcodes,
    columns=np.arange(1, mito_length + 1),
)
total_coverage

In [ ]:
for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]


In [ ]:
total_coverage

In [ ]:
total_coverage.mean(axis = 1)

In [ ]:
# exclude low coverage cells from variant calling

cell_barcodes = total_coverage.index[
    total_coverage.mean(axis=1) > low_coverage_threshold
]


In [ ]:
for nt in base_coverage_dict:
    base_coverage_dict[nt] = (
        base_coverage_dict[nt][0].loc[cell_barcodes, :],
        base_coverage_dict[nt][1].loc[cell_barcodes, :],
    )
total_coverage = total_coverage.loc[cell_barcodes, :]


In [ ]:
reference_file = MGATK_OUT_DIR + mito_genome + "_refAllele.txt"

aggregated_genotype = pd.DataFrame(
  np.zeros((4, len(mito_genome))),
  index = list("ATCG"),
  columns = np.arange(1, len(mito_genome) + 1)
)

In [ ]:
nt = "A"

In [ ]:
fwd_base_df, rev_base_df = base_coverage_dict[nt]
fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

In [ ]:
masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))
fwd_base_sum[masking], rev_base_sum[masking] = 0, 0


In [ ]:
aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
for nt in base_coverage_dict:
    # sum across cells for each strand separately
    fwd_base_df, rev_base_df = base_coverage_dict[nt]
    fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

    # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
    masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
    fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

    # sum across strands
    aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

In [ ]:
aggregated_genotype

In [ ]:
ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]


In [ ]:
ref_set[:4]

In [ ]:
ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]

In [ ]:
ref_N_positions

In [ ]:

ref_set = set([(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters])
ref_dict = dict(ref_set)


In [ ]:
aggregated_genotype

In [ ]:
a = pd.DataFrame({'a': [1, 0, 3], 'b': [4, 5, 6]})

In [ ]:
a

In [ ]:
np.where(a > 0)

In [ ]:
aggregated_genotype

In [ ]:
non_zero_idx = np.where(aggregated_genotype > 0)


In [ ]:
non_zero_bases = [letters[i] for i in non_zero_idx[0]]
non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]

In [ ]:
non_zero_bases[:4]

In [ ]:
non_zero_pos[:4]

In [ ]:
observed_set = list(zip(non_zero_pos, non_zero_base))

In [ ]:
observed_set[:5]

In [ ]:
bserved_set = set([x for x in observed_set if x[0] not in ref_N_positions])  # disregard position

In [ ]:
bserved_set[0:10]

In [ ]:
list(bserved_set)[0:10]

In [ ]:
list(ref_set)[0:10]

In [ ]:
variant_set = set(observed_set) - set(ref_set)

In [ ]:
variant_set

In [ ]:
ref_dict[1]

In [ ]:
variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0])

In [ ]:
non_zero_idx[1]

In [ ]:
masking

In [ ]:
reference_file

In [ ]:
# cell positional variants
variants = gather_possible_variants(
    base_coverage_dict, MGATK_OUT_DIR + mito_genome + "_refAllele.txt"
)


In [ ]:
len(variants)

In [ ]:
variant_names = ["{}{}>{}".format(x[0], x[1], x[2]) for x in variants]


In [ ]:
variant_names

In [ ]:
# build two <cell by variant tables>, one for each strand
total_coverage_variant_df = []
fwd_cell_variant_df, rev_cell_variant_df = [], []


In [ ]:
total_coverage[2]


In [ ]:
base_coverage_dict["A"][0][3].values

In [ ]:
for i, var in enumerate(variants):
    var_name = variant_names[i]
    pos, base = var[0], var[2]
    total_coverage_variant_df.append(total_coverage[pos])
    fwd_cell_variant_df.append(base_coverage_dict[base][0][pos].values)
    rev_cell_variant_df.append(base_coverage_dict[base][1][pos].values)


In [ ]:
fwd_cell_variant_df

In [ ]:
total_coverage_variant_df = pd.DataFrame(np.array(total_coverage_variant_df).T, index=cell_barcodes, columns=variant_names)


In [ ]:
total_coverage_variant_df

In [ ]:
fwd_cell_variant_df = pd.DataFrame(np.array(fwd_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
rev_cell_variant_df = pd.DataFrame(np.array(rev_cell_variant_df).T, index=cell_barcodes, columns=variant_names)


In [ ]:
all_cell_variant_df = fwd_cell_variant_df + rev_cell_variant_df

In [ ]:
all_cell_variant_df

In [ ]:
heteroplasmic_df = all_cell_variant_df / total_coverage_variant_df

In [ ]:
heteroplasmic_df

In [ ]:
mask_idx = (fwd_cell_variant_df + rev_cell_variant_df) == 0  # set 0 on both strands to nan to exclude from correlation calculation
fwd_cell_variant_df[mask_idx] = np.nan
rev_cell_variant_df[mask_idx] = np.nan


In [ ]:
fwd_cell_variant_df

In [ ]:
variant_strand_corr = fwd_cell_variant_df.corrwith(rev_cell_variant_df).round(3)

In [ ]:
variant_strand_corr

In [ ]:
variant_mean = all_cell_variant_df.sum() / total_coverage_variant_df.sum()
variant_var = heteroplasmic_df.var()


In [ ]:
variant_vmr = variant_var / (variant_mean + 0.00000000001)

In [ ]:
variant_vmr

In [ ]:
variants

In [ ]:
((fwd_cell_variant_df >= 2) & (rev_cell_variant_df >= 2)).sum()

In [ ]:
heteroplasmic_df

In [ ]:
# compute other summary stats
variant_positon = [x[0] for x in variants]
variant_nucleotide = ["{}>{}".format(x[1], x[2]) for x in variants]
variant_n_cells_conf_detected = ((fwd_cell_variant_df >= 2) & (rev_cell_variant_df >= 2)).sum()
variant_n_cells_over_5 = (heteroplasmic_df >= 0.05).sum()
variant_n_cells_over_10 = (heteroplasmic_df >= 0.1).sum()
variant_n_cells_over_20 = (heteroplasmic_df >= 0.2).sum()
variant_n_cells_over_95 = (heteroplasmic_df >= 0.95).sum()
max_heteroplasmy = heteroplasmic_df.max()
variant_mean_coverage = total_coverage_variant_df.mean()

In [ ]:
# pack summary stats
variant_output = pd.DataFrame(
    [
        variant_positon,
        variant_nucleotide,
        variant_names,
        variant_vmr,
        variant_mean,
        variant_var,
        variant_n_cells_conf_detected,
        variant_n_cells_over_5,
        variant_n_cells_over_10,
        variant_n_cells_over_20,
        variant_n_cells_over_95,
        max_heteroplasmy,
        variant_strand_corr,
        variant_mean_coverage,
    ]
).T
variant_output.columns = ["position", "nucleotide", "variant", "vmr", "mean", "variance", "n_cells_conf_detected", "n_cells_over_5", "n_cells_over_10", "n_cells_over_20", "n_cells_over_95", "max_heteroplasmy", "strand_correlation", "mean_coverage"]
variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]] = variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]].astype(float)


In [ ]:
variant_output